# Rebuild manuscript tables `tab:ring_summary` and `tab:ppc_all`

Extracts the selected PR-retrieval rows of Numpaque et al. (2026) from versioned
dynesty results under `pipeline/kepler_51/results/exorings/`.

Emits **only the `tabular*` bodies** (header + data rows). Caption, label,
footnotes, `\table*` wrapper, and font/spacing knobs stay in the manuscript so
writers keep editorial control. Typical usage in the `.tex` file:

```latex
\begin{table*}[t!]
\centering
\caption{...}
\label{tab:ring_summary}
\scriptsize
\setlength{\tabcolsep}{0pt}
\input{tab_ring_summary.tex}
% optional note / footnote here
\end{table*}
```

- **`tab_ring_summary.tex`** — posterior medians ± 68% CI (from `*_meta.json`), with
  fixed nuisance/input parameters marked $\dagger$. Derived $\rho_p$ and $R_\oplus$
  from $p$ using $M_b=6.7\,M_\oplus$, $M_d=5.6\,M_\oplus$ (Masuda+2024) and
  $R_\star=0.869\,R_\odot$ (Berger+2023).
- **`tab_ppc_all.tex`** — $W_1$ / $E$ PPC metrics recomputed from the `ppc` arrays in
  each `.npz` against the photometric TTV posteriors. Includes the row that was
  missing in the handwritten manuscript: Kepler-51 b,
  $\mathcal{L}=(\delta,T_{14},\rho_{\star,\rm obs})$, All flexible (13 rows total).

Each regenerated ``.tex`` is stamped with a ``% run_version: YYYYMMDDHH`` comment
derived from the newest input ``.npz`` mtime, so writers can trace which campaign
produced the numbers.


In [1]:
import sys, json, pathlib
import numpy as np
from datetime import datetime

_HERE = pathlib.Path.cwd()
_cands = [_HERE, *_HERE.parents]
_REPO = next(
    (c for c in _cands if (c / "pipeline").is_dir() and (c / "exorings").is_dir()),
    _HERE.parent,
)
for _p in (str(_REPO), str(_REPO / "pipeline")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import photoring as pr
from photoring.plotting import ppc_stats_1d, OBS_META

RESULTS = _REPO / "pipeline" / "kepler_51" / "results" / "exorings"
REFERENCE = _REPO / "papers" / "kepler51" / "reference_runs"
OUT_RING = _REPO / "papers" / "kepler51" / "tab_ring_summary.tex"
OUT_PPC = _REPO / "papers" / "kepler51" / "tab_ppc_all.tex"
paths = pr.CasePaths("kepler_51", pipeline_dir=_REPO / "pipeline")


def run_version_id(files):
    """YYYYMMDDHH stamp from the newest file mtime (campaign identifier)."""
    files = [pathlib.Path(f) for f in files if f is not None and pathlib.Path(f).exists()]
    if not files:
        return "unknown"
    newest = max(files, key=lambda p: p.stat().st_mtime)
    return datetime.fromtimestamp(newest.stat().st_mtime).strftime("%Y%m%d%H")


print("repo:", _REPO)
print("results:", RESULTS, "(n=", len(list(RESULTS.glob("*_meta.json"))), ")")


repo: /Users/jzuluaga/dev/PRisma
results: /Users/jzuluaga/dev/PRisma/pipeline/kepler_51/results/exorings (n= 82 )


## Shared row catalog (same order as the manuscript)


In [2]:
def run_stem(planet, kde, flags):
    base = (
        f"kepler_51_{planet}_NS_exorings_kde_{kde}"
        f"_nlive1200_dlogz0.01_NKDE5000_seed2026"
    )
    return f"{base}_{flags}" if flags else base


# (planet, kde_tag, flags, config_label, show_kde, midrule_after)
# NOTE: includes the PPC row missing from the handwritten tab:ppc_all —
#   b / delta-T14-rho_obs / All flexible
ROWS = [
    # ── Kepler-51 b ──────────────────────────────────────────────────────────
    ("b", "delta-rho_obs",     "rhoFREE_bFREE_tauFREE_pFREE", "All flexible",
     True,  False),
    ("b", "delta-rho_obs",     "rhoFREE",                     r"$\rho_{\star,\rm true}$",
     False, False),
    ("b", "delta-rho_obs",     "",                            "All fixed",
     False, True),

    ("b", "delta-T14-rho_obs", "rhoFREE_bFREE_tauFREE_pFREE", "All flexible",  # was missing in tab:ppc_all
     True,  False),
    ("b", "delta-T14-rho_obs", "rhoFREE_bFREE_pFREE",         r"$\rho_{\star,\rm true}+b+p$",
     False, False),
    ("b", "delta-T14-rho_obs", "rhoFREE_bFREE",               r"$\rho_{\star,\rm true}+b$",
     False, True),

    ("b", "delta-T14-T23",     "rhoFREE_bFREE_tauFREE_pFREE", "All flexible",
     True,  False),

    # ── Kepler-51 d ──────────────────────────────────────────────────────────
    ("d", "delta-rho_obs",     "rhoFREE_bFREE_tauFREE_pFREE", "All flexible",
     True,  False),
    ("d", "delta-rho_obs",     "",                            "All fixed",
     False, True),

    ("d", "delta-T14-rho_obs", "rhoFREE_bFREE_tauFREE_pFREE", "All flexible",
     True,  False),
    ("d", "delta-T14-rho_obs", "rhoFREE_bFREE_pFREE",         r"$\rho_{\star,\rm true}+b+p$",
     False, False),
    ("d", "delta-T14-rho_obs", "rhoFREE_pFREE",               r"$\rho_{\star,\rm true}+p$",
     False, True),

    ("d", "delta-T14-T23",     "rhoFREE_bFREE_tauFREE_pFREE", "All flexible",
     True,  False),
]

KDE_LATEX = {
    "delta-rho_obs":     r"$\delta, \rho_{\star,\rm obs}$",
    "delta-T14-rho_obs": r"$\delta, T_{14}, \rho_{\star,\rm obs}$",
    "delta-T14-T23":     r"$\delta, T_{14}, T_{23}$",
}

# Observables that enter L_KDE (bold in tab:ppc_all). b_obs is always out-of-sample.
IN_SAMPLE = {
    "delta-rho_obs":     {"delta", "rho_obs"},
    "delta-T14-rho_obs": {"delta", "T14", "rho_obs"},
    "delta-T14-T23":     {"delta", "T14", "T23"},
}

PPC_KEYS = ["delta", "T14", "T23", "rho_obs", "b_obs"]

print(f"{len(ROWS)} selected rows")
for planet, kde, flags, label, *_ in ROWS:
    stem = run_stem(planet, kde, flags)
    hit = (RESULTS / f"{stem}_meta.json").exists()
    print(f"  [{'OK' if hit else 'MISSING'}] {planet} | {kde:18s} | {label}")


13 selected rows
  [OK] b | delta-rho_obs      | All flexible
  [OK] b | delta-rho_obs      | $\rho_{\star,\rm true}$
  [OK] b | delta-rho_obs      | All fixed
  [OK] b | delta-T14-rho_obs  | All flexible
  [OK] b | delta-T14-rho_obs  | $\rho_{\star,\rm true}+b+p$
  [OK] b | delta-T14-rho_obs  | $\rho_{\star,\rm true}+b$
  [OK] b | delta-T14-T23      | All flexible
  [OK] d | delta-rho_obs      | All flexible
  [OK] d | delta-rho_obs      | All fixed
  [OK] d | delta-T14-rho_obs  | All flexible
  [OK] d | delta-T14-rho_obs  | $\rho_{\star,\rm true}+b+p$
  [OK] d | delta-T14-rho_obs  | $\rho_{\star,\rm true}+p$
  [OK] d | delta-T14-T23      | All flexible


## Helpers — ring-summary formatting


In [3]:
# Physical constants (cgs) — same numbers as papers/kepler51/PRisma-PRErrors.ipynb
M_SUN = 1.98847e33
R_SUN = 6.957e10
M_EARTH = 5.9722e27
R_EARTH = 6.371e8
R_STAR = 0.869 * R_SUN   # Berger+2023
MASS = {"b": 6.7 * M_EARTH, "d": 5.6 * M_EARTH}  # Masuda+2024


def p_to_Rp_earth(p):
    return float(p) * R_STAR / R_EARTH


def p_to_rho_p(p, planet):
    Rp = float(p) * R_STAR
    return MASS[planet] / ((4.0 / 3.0) * np.pi * Rp**3)


def asym(med, lo, hi, prec):
    fmt = f"{{:.{prec}f}}"
    return (
        f"${fmt.format(med)}"
        f"_{{-{fmt.format(med - lo)}}}"
        f"^{{+{fmt.format(hi - med)}}}$"
    )


def fixed(val, prec):
    return f"${val:.{prec}f}^\\dagger$"


def fmt_p_cell(med, lo, hi, free, prec_p=2, prec_R=2):
    if not free:
        Rp = p_to_Rp_earth(med)
        return f"${med:.{prec_p}f}^\\dagger\\;({Rp:.{prec_R}f}^\\dagger)$"
    Rp_med, Rp_lo, Rp_hi = map(p_to_Rp_earth, (med, lo, hi))
    fmt_p = f"{{:.{prec_p}f}}"
    fmt_R = f"{{:.{prec_R}f}}"
    return (
        f"${fmt_p.format(med)}"
        f"_{{-{fmt_p.format(med - lo)}}}"
        f"^{{+{fmt_p.format(hi - med)}}}"
        f"\\;({fmt_R.format(Rp_med)}"
        f"_{{-{fmt_R.format(Rp_med - Rp_lo)}}}"
        f"^{{+{fmt_R.format(Rp_hi - Rp_med)}}})$"
    )


def fmt_rho_p_cell(med_p, lo_p, hi_p, free, planet, prec=2):
    rho_med = p_to_rho_p(med_p, planet)
    if not free:
        return f"${rho_med:.{prec}f}$"
    rho_at_lo = p_to_rho_p(lo_p, planet)
    rho_at_hi = p_to_rho_p(hi_p, planet)
    return asym(rho_med, rho_at_hi, rho_at_lo, prec)


def load_meta(stem):
    for root in (RESULTS, REFERENCE):
        path = root / f"{stem}_meta.json"
        if path.exists():
            return json.loads(path.read_text()), path
    raise FileNotFoundError(stem)


def load_ppc(stem):
    for root in (RESULTS, REFERENCE):
        path = root / f"{stem}.npz"
        if path.exists():
            return np.load(path)["ppc"], path
    raise FileNotFoundError(stem + ".npz")


def get_stat(meta, name):
    free_key = {
        "p": "P_FREE", "tau": "TAU_FREE",
        "rho_true": "RHO_TRUE_FREE", "b": "B_FREE",
    }.get(name)
    free = True if free_key is None else bool(meta.get(free_key, False))
    if free and f"stat_{name}_median" in meta:
        return (meta[f"stat_{name}_median"], meta[f"stat_{name}_p16"],
                meta[f"stat_{name}_p84"], True)
    fixed_map = {
        "p": "P_FIXED_VALUE", "tau": "TAU_FIXED",
        "rho_true": "RHO_TRUE_FIXED", "b": "B_FIXED",
    }
    val = float(meta[fixed_map[name]])
    return val, val, val, False


def ring_cells(planet, meta):
    fe, ir, th = get_stat(meta, "fe"), get_stat(meta, "ir"), get_stat(meta, "theta")
    p, tau = get_stat(meta, "p"), get_stat(meta, "tau")
    rho, b = get_stat(meta, "rho_true"), get_stat(meta, "b")

    if not p[3]:
        p_cell = f"${p[0]:.4f}^\\dagger\\;({p_to_Rp_earth(p[0]):.2f}^\\dagger)$"
    else:
        prec_p = 3 if p[0] < 0.05 or abs(p[0] - 0.078) < 0.01 or abs(p[0] - 0.069) < 0.01 else 2
        p_cell = fmt_p_cell(p[0], p[1], p[2], True, prec_p=prec_p)

    if not rho[3]:
        rho_cell = fixed(rho[0], 3)
    else:
        span = rho[2] - rho[1]
        rho_cell = asym(rho[0], rho[1], rho[2], 3 if span < 0.2 else 2)

    b_cell = fixed(b[0], 4) if not b[3] else asym(b[0], b[1], b[2], 2)
    tau_cell = fixed(tau[0], 2) if not tau[3] else asym(tau[0], tau[1], tau[2], 2)

    rp = p_to_rho_p(p[0], planet)
    rho_p_prec = 3 if rp < 0.1 else 2
    rho_p_cell = fmt_rho_p_cell(p[0], p[1], p[2], p[3], planet, prec=rho_p_prec)

    return [
        asym(fe[0], fe[1], fe[2], 2),
        asym(ir[0], ir[1], ir[2], 2),
        asym(th[0], th[1], th[2], 2),
        p_cell, tau_cell, rho_cell, b_cell, rho_p_cell,
    ]


## Load runs + compute PPC metrics


In [4]:
_ttv_cache = {}


def ttv_for(planet):
    if planet not in _ttv_cache:
        _ttv_cache[planet] = pr.load_case_data(paths, planet)["ttv"]
    return _ttv_cache[planet]


def compute_ppc_stats(planet, ppc):
    """W1 / E against photometric TTV posteriors (delta in ppm)."""
    ttv = ttv_for(planet)
    out = {}
    for key in PPC_KEYS:
        meta = OBS_META[key]
        scale = meta["scale"]
        emp = np.asarray(ttv[meta["df_col"]], float) * scale
        pred = np.asarray(ppc[:, meta["ppc_col"]], float) * scale
        out[key] = ppc_stats_1d(emp, pred)
    return out


def fmt_sig(x, sig=3):
    if not np.isfinite(x):
        return "nan"
    return f"{x:.{sig}g}"


def fmt_ppc_cell(w1, e, bold=False):
    text = f"{fmt_sig(w1)}\\,/\\,{fmt_sig(e)}"
    return f"\\textbf{{{text}}}" if bold else text


loaded = []
for planet, kde, flags, label, show_kde, midrule in ROWS:
    stem = run_stem(planet, kde, flags)
    meta, meta_path = load_meta(stem)
    ppc, ppc_path = load_ppc(stem)
    ppc_stats = compute_ppc_stats(planet, ppc)
    loaded.append(dict(
        planet=planet, kde=kde, flags=flags, label=label,
        show_kde=show_kde, midrule=midrule,
        meta=meta, meta_path=meta_path, ppc_path=ppc_path,
        ring_cells=ring_cells(planet, meta),
        ppc_stats=ppc_stats, logz=meta["logz"],
    ))
    d = ppc_stats["delta"]
    print(f"{planet} {label:28s} lnZ={meta['logz']:+7.3f}  "
          f"PPC δ W1={d['W1']:.3g} E={d['E']:.3g}")


b All flexible                 lnZ= +2.236  PPC δ W1=3.97 E=0.448
b $\rho_{\star,\rm true}$      lnZ= +1.602  PPC δ W1=1.3 E=0.142
b All fixed                    lnZ= +1.542  PPC δ W1=3.67 E=0.384
b All flexible                 lnZ= +1.657  PPC δ W1=5.09 E=0.621
b $\rho_{\star,\rm true}+b+p$  lnZ= +2.627  PPC δ W1=4.73 E=0.554
b $\rho_{\star,\rm true}+b$    lnZ= +1.323  PPC δ W1=4.07 E=0.464
b All flexible                 lnZ= +3.400  PPC δ W1=8.34 E=0.969
d All flexible                 lnZ= +2.307  PPC δ W1=22.3 E=1.43
d All fixed                    lnZ= +1.682  PPC δ W1=19.1 E=1.21
d All flexible                 lnZ= +1.572  PPC δ W1=20.4 E=1.31
d $\rho_{\star,\rm true}+b+p$  lnZ= +1.962  PPC δ W1=18.8 E=1.39
d $\rho_{\star,\rm true}+p$    lnZ= +1.653  PPC δ W1=18.5 E=1.14
d All flexible                 lnZ= +2.253  PPC δ W1=20.1 E=1.21


## Emit `tab_ring_summary.tex` (tabular* only)


In [5]:
HEADER_RING = r"""\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}} c @{\hspace{0.3em}} c @{\hspace{0.3em}} c @{\hspace{0.3em}} c @{\hspace{0.3em}} c @{\hspace{0.3em}} c @{\hspace{0.3em}} c @{\hspace{0.3em}} c @{\hspace{0.3em}} c @{\hspace{0.3em}} c @{\hspace{0.3em}} c @{}}
\toprule
& Observable set & Run config. & \multicolumn{7}{c}{Input selection} & Other\\
\cmidrule(lr){4-10}
Planet &  ($\mathcal{L}_{\rm KDE}$) & Nuis.+Input
  & $f_e\;[R_p]$ & $i_R\;[^\circ]$ & $\theta_R\;[^\circ]$
  & $p\;[R_\star]\;(R_\oplus)$ & $\tau$
  & $\rho_{\star,\rm true}\;[\rm g\,cm^{-3}]$ & $b$ & $\rho_p\;[\rm g\,cm^{-3}]$ \\
\midrule
"""

FOOTER_RING = r"""\bottomrule
\end{tabular*}
"""

_npz_paths = []
for r in loaded:
    meta_p = pathlib.Path(r["meta_path"])
    npz_p = meta_p.with_name(meta_p.name.replace("_meta.json", ".npz"))
    _npz_paths.append(npz_p if npz_p.exists() else meta_p)
RUN_VERSION = run_version_id(_npz_paths)
print("RUN_VERSION:", RUN_VERSION)


def emit_table(loaded_rows, header, footer, cell_key, n_cols_cmidrule, out_path,
               version=None):
    """Write only the tabular(*) body — no table*/caption/label/notes."""
    lines = []
    if version:
        lines.append(f"% run_version: {version}")
    lines.append(header)
    prev_planet = None
    for i, row in enumerate(loaded_rows):
        planet = row["planet"]
        if planet != prev_planet:
            if prev_planet is not None:
                lines.append(r"\midrule")
                lines.append("")
            n_planet = sum(1 for r in loaded_rows if r["planet"] == planet)
            planet_cell = (
                rf"\multirow{{{n_planet}}}{{*}}{{\rotatebox{{90}}{{\exoplanet{{Kepler-51}}{{{planet}}}}}}}"
            )
            prev_planet = planet
            lines.append(f"% ======================== PLANET {planet.upper()} ========================")
        else:
            planet_cell = ""

        if row["show_kde"]:
            n_kde = 1
            for j in range(i + 1, len(loaded_rows)):
                if loaded_rows[j]["planet"] != planet or loaded_rows[j]["kde"] != row["kde"]:
                    break
                n_kde += 1
            kde_tex = KDE_LATEX[row["kde"]]
            kde_cell = rf"\multirow{{{n_kde}}}{{*}}{{{kde_tex}}}" if n_kde > 1 else kde_tex
        else:
            kde_cell = ""

        body = "\n    & ".join([row["label"], *row[cell_key]])
        lines.append(f"{planet_cell}\n  & {kde_cell}\n    & {body} \\\\")
        if row["midrule"]:
            lines.append(rf"\cmidrule{{2-{n_cols_cmidrule}}}")
            lines.append("")
    lines.append(footer)
    tex = "\n".join(lines)
    out_path.write_text(tex)
    print(f"Wrote {out_path}  ({len(tex.splitlines())} lines, tabular* only; version={version})")
    return tex


tex_ring = emit_table(loaded, HEADER_RING, FOOTER_RING, "ring_cells", 11, OUT_RING,
                      version=RUN_VERSION)


RUN_VERSION: 2026073017
Wrote /Users/jzuluaga/dev/PRisma/papers/kepler51/tab_ring_summary.tex  (168 lines, tabular* only; version=2026073017)


## Emit `tab_ppc_all.tex` (tabular* only)

Includes **all 13** selected scenarios (the handwritten manuscript omitted
Kepler-51 b / $\delta,T_{14},\rho_{\star,\rm obs}$ / All flexible). In-sample
observables (members of $\mathcal{L}_{\rm KDE}$) are bold; $b_{\rm obs}$ is
always out-of-sample. Caption and footnote stay in the manuscript.


In [6]:
HEADER_PPC = r"""\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}} c c c c c c c c @{}}
\toprule
Planet &
Observable set &
Run config. &
$\delta$ [ppm] &
$T_{14}$ [h] &
$T_{23}$ [h] &
$\rho_{\star,\rm obs}$ [g\,cm$^{-3}$] &
$b_{\rm obs}$\\
&  ($\mathcal{L}_{\rm KDE}$) & Nuis.+Input & & & & & \\
\midrule
"""

FOOTER_PPC = r"""\bottomrule
\end{tabular*}
"""

# Attach ppc cell list onto each row
for row in loaded:
    ins = IN_SAMPLE[row["kde"]]
    row["ppc_cells"] = [
        fmt_ppc_cell(row["ppc_stats"][k]["W1"], row["ppc_stats"][k]["E"],
                     bold=(k in ins))
        for k in PPC_KEYS
    ]

tex_ppc = emit_table(loaded, HEADER_PPC, FOOTER_PPC, "ppc_cells", 8, OUT_PPC,
                      version=RUN_VERSION)
print(tex_ppc)


Wrote /Users/jzuluaga/dev/PRisma/papers/kepler51/tab_ppc_all.tex  (132 lines, tabular* only; version=2026073017)
% run_version: 2026073017
\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}} c c c c c c c c @{}}
\toprule
Planet &
Observable set &
Run config. &
$\delta$ [ppm] &
$T_{14}$ [h] &
$T_{23}$ [h] &
$\rho_{\star,\rm obs}$ [g\,cm$^{-3}$] &
$b_{\rm obs}$\\
&  ($\mathcal{L}_{\rm KDE}$) & Nuis.+Input & & & & & \\
\midrule

% ======================== PLANET B ========================
\multirow{7}{*}{\rotatebox{90}{\exoplanet{Kepler-51}{b}}}
  & \multirow{3}{*}{$\delta, \rho_{\star,\rm obs}$}
    & All flexible
    & \textbf{3.97\,/\,0.448}
    & 0.328\,/\,0.603
    & 0.369\,/\,0.642
    & \textbf{0.0267\,/\,0.0647}
    & 0.0542\,/\,0.136 \\

  & 
    & $\rho_{\star,\rm true}$
    & \textbf{1.3\,/\,0.142}
    & 0.455\,/\,0.795
    & 0.517\,/\,0.853
    & \textbf{0.00745\,/\,0.0171}
    & 0.0116\,/\,0.0333 \\

  & 
    & All fixed
    & \textbf{3.67\,/\,0.384}
    & 0.471\,/\,0.945
   

## Sanity check — missing row vs neighbours


In [7]:
# Highlight the newly included b / δ,T14,ρ / All flexible PPC row
miss = next(
    r for r in loaded
    if r["planet"] == "b" and r["kde"] == "delta-T14-rho_obs"
    and r["flags"] == "rhoFREE_bFREE_tauFREE_pFREE"
)
print("Inserted row: Kepler-51 b | δ,T14,ρ★,obs | All flexible")
print(f"  source: {miss['ppc_path'].name}")
print(f"  lnZ    = {miss['logz']:+.3f}")
for k in PPC_KEYS:
    s = miss["ppc_stats"][k]
    tag = "IN " if k in IN_SAMPLE[miss["kde"]] else "out"
    print(f"  [{tag}] {k:8s}  W1={s['W1']:.4g}  E={s['E']:.4g}")


Inserted row: Kepler-51 b | δ,T14,ρ★,obs | All flexible
  source: kepler_51_b_NS_exorings_kde_delta-T14-rho_obs_nlive1200_dlogz0.01_NKDE5000_seed2026_rhoFREE_bFREE_tauFREE_pFREE.npz
  lnZ    = +1.657
  [IN ] delta     W1=5.085  E=0.6213
  [IN ] T14       W1=0.006106  E=0.02907
  [out] T23       W1=0.004627  E=0.0239
  [IN ] rho_obs   W1=0.0148  E=0.03635
  [out] b_obs     W1=0.01777  E=0.05196
